# Experiment 01 — Data Audit

Notebook kiểm tra dữ liệu đầu vào của benchmark ACME Insurance
theo bài báo Sequeda et al. (2024).

Nội dung:
- Kiểm kê dữ liệu CSV và giá trị thiếu.
- Kiểm tra SQL DDL.
- Thống kê OWL Ontology và R2RML Mapping.
- Xác nhận 43 câu hỏi và các truy vấn Reference.

Notebook chỉ đọc dữ liệu, không sửa dữ liệu gốc.

In [6]:
from pathlib import Path
import re
import pandas as pd
from rdflib import Graph, Namespace, RDF, OWL, URIRef

PROJECT_DIR = Path.cwd().resolve()

if not (PROJECT_DIR / "data" / "ACME_Insurance").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "ACME_Insurance"
CSV_DIR = DATA_DIR / "data"

DDL_FILE = DATA_DIR / "DDL" / "ACME_small.ddl"
ONTOLOGY_FILE = DATA_DIR / "ontology" / "insurance.ttl"
MAPPING_FILE = CSV_DIR / "data.world_P&C_Insurance_Ontology_V1.r2rml"
BENCHMARK_FILE = DATA_DIR / "investigation" / "acme-benchmark.ttl"

files = [CSV_DIR, DDL_FILE, ONTOLOGY_FILE, MAPPING_FILE, BENCHMARK_FILE]
assert all(path.exists() for path in files), "Thiếu dữ liệu đầu vào"

print("Project:", PROJECT_DIR)
print("Environment: OK")

Project: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt
Environment: OK


In [7]:
csv_files = sorted(CSV_DIR.glob("*.csv"))

summary = []

for file in csv_files:
    df = pd.read_csv(file)

    summary.append({
        "Table": file.stem,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing": int(df.isna().sum().sum())
    })

summary_df = pd.DataFrame(summary)

print("Total tables:", len(summary_df))
print("Total rows:", summary_df["Rows"].sum())
print("Total missing values:", summary_df["Missing"].sum())

display(summary_df)

Total tables: 29
Total rows: 977
Total missing values: 3988


,Table,Rows,Columns,Missing
0,Agreement,2,5,6
1,Agreement_Assessment,2,2,0
2,Agreement_Party_Role,4,5,8
3,Assesment_Party_Role,2,5,4
4,Assessment,2,5,4
5,Assessment_Result,2,3,0
6,Catastrophe,4,5,12
7,Claim,2,14,14
8,Claim_Amount,8,7,32
9,Claim_Coverage,2,3,0


In [8]:
ddl_text = DDL_FILE.read_text(encoding="utf-8-sig")

ddl_tables = re.findall(
    r"CREATE TABLE\s+(\w+)",
    ddl_text,
    flags=re.IGNORECASE
)

csv_names = {file.stem.lower() for file in csv_files}
missing_tables = [
    name for name in ddl_tables
    if name.lower() not in csv_names
]

print("Tables in DDL:", len(ddl_tables))
print("Missing CSV tables:", missing_tables)

Tables in DDL: 13
Missing CSV tables: []


In [9]:
ontology = Graph()
ontology.parse(ONTOLOGY_FILE, format="turtle")

mapping = Graph()
mapping.parse(MAPPING_FILE, format="turtle")

RR = Namespace("http://www.w3.org/ns/r2rml#")

print("=== OWL ONTOLOGY ===")
print("Triples:", len(ontology))
print("Classes:", len(set(ontology.subjects(RDF.type, OWL.Class))))
print("Object Properties:", len(set(ontology.subjects(RDF.type, OWL.ObjectProperty))))
print("Datatype Properties:", len(set(ontology.subjects(RDF.type, OWL.DatatypeProperty))))

print("\n=== R2RML MAPPING ===")
print("Triples:", len(mapping))
print("Triples Maps:", len(set(mapping.subjects(RDF.type, RR.TriplesMap))))

=== OWL ONTOLOGY ===
Triples: 164
Classes: 11
Object Properties: 10
Datatype Properties: 18

=== R2RML MAPPING ===
Triples: 561
Triples Maps: 62


In [10]:
benchmark = Graph()
benchmark.parse(BENCHMARK_FILE, format="turtle")

QA = Namespace("http://models.data.world/benchmarks/QandA#")

extra_question = "Return all the claims by claim number that were closed in 2019"

questions = []

for inquiry in sorted(benchmark.subjects(RDF.type, QA.Inquiry), key=str):
    question = str(benchmark.value(inquiry, QA.prompt))

    if question != extra_question:
        questions.append({
            "Inquiry URI": str(inquiry),
            "Question": question
        })

questions_df = pd.DataFrame(questions)
questions_df.insert(0, "ID", range(1, len(questions_df) + 1))

assert len(questions_df) == 43

print("Total questions:", len(questions_df))
display(questions_df.head())

Total questions: 43


,ID,Inquiry URI,Question
0,1,https://templates.data.world/IQ_03fc5a5571db11...,What is the loss ratio of each policy and agen...
1,2,https://templates.data.world/IQ_0d87a2a016062b...,Return all the policies we have by policy numb...
2,3,https://templates.data.world/IQ_1e19282f42eaea...,"What are the loss payment, loss reserve, expen..."
3,4,https://templates.data.world/IQ_1e57233cbc8a71...,"What are the total loss, which is the sum of l..."
4,5,https://templates.data.world/IQ_205ac9d7d19a51...,Return agents and the policy they have sold th...


In [11]:
DWT = Namespace("https://templates.data.world/")

references = []

for _, row in questions_df.iterrows():
    inquiry = URIRef(row["Inquiry URI"])
    refs = list(benchmark.objects(inquiry, QA.expects))

    references.append({
        "ID": row["ID"],
        "SQL": sum((ref, RDF.type, DWT.SqlQuery) in benchmark for ref in refs),
        "SPARQL": sum((ref, RDF.type, DWT.SparqlQuery) in benchmark for ref in refs)
    })

reference_df = pd.DataFrame(references)

print("Questions:", len(reference_df))
print("SQL References:", reference_df["SQL"].sum())
print("SPARQL References:", reference_df["SPARQL"].sum())

assert (reference_df[["SQL", "SPARQL"]] >= 1).all().all()

display(reference_df.head())

Questions: 43
SQL References: 44
SPARQL References: 43


,ID,SQL,SPARQL
0,1,1,1
1,2,1,1
2,3,1,1
3,4,1,1
4,5,1,1
